# 06 · Hardware session 2: replication and a real test of Lemma 3

**Settings:** Accelerator *None*, Internet **on**, Secrets `IBM_QUANTUM_TOKEN` (and optionally `IBM_QUANTUM_INSTANCE`).
**Inputs:** `bbd-code` (revised), the outputs of notebooks **01**, **02**, **03** (session 1: its `jobs.json`
holds the qubit path and the job id) and **05** (the two new hardware models).

One job, sized to the QPU time you have left: the notebook reads your remaining Open Plan seconds and picks
512, 384, 256 or 128 shots (512 shots need about 3.4 QPU-minutes). It uses the session-1 qubit path and 304 of
the 600 session-1 test inputs (every other input of each class, 38 per class) plus the 56 calibration inputs:

* the LA model on layout A and on layout B again: replication, a paired test of A against B, and a paired test of
  session 1 against session 2 on the same inputs;
* a post-hoc model at s = 2.35, just above the rho = 100 threshold. Lemma 3 predicts that a measured
  contraction alpha < 0.98 makes the most frequent class unreachable on raw hardware and that the global rescale
  restores it, so H5 becomes a real test;
* the entanglement-free circuit (Q-noent) on layout A, to separate CZ-gate noise from readout noise.

Run once with `DRY_RUN = True` (simulated kingston, no token, no cost), then with `DRY_RUN = False`.
If the queue outlasts the session, commit, attach this notebook's output and set `RESUME_JOB`.

In [ ]:
# Pinned: these are the versions the hardware code was tested with (SamplerV2 API).
%pip install -q "qiskit==2.6.1" "qiskit-ibm-runtime==0.51.0" "qiskit-aer==0.18.0" pylatexenc

In [ ]:
# --- bootstrap: find the bbd-code dataset, copy it to a writable folder, import it
import glob, os, shutil, sys
# only the bbd-code dataset counts: attached notebook outputs never contain the code (it is copied to /tmp)
hits = [h for h in sorted(glob.glob("/kaggle/input/**/bbd/__init__.py", recursive=True)) if "/bbd_code/" not in h]
hits = hits or sorted(glob.glob("./bbd/__init__.py"))
if not hits:
    raise FileNotFoundError("Attach the 'bbd-code' dataset to this notebook (KAGGLE_INSTRUCTIONS.md, step 2).")
if len(hits) > 1:
    print("warning: several copies of the code are attached; using", hits[0])
SRC = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
CODE = "/tmp/bbd_code" if os.path.isdir("/kaggle/working") else SRC
if CODE != SRC:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results", "work"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
import bbd
print("bbd", bbd.__version__, "loaded from", SRC)

In [ ]:
DRY_RUN = True
BACKEND = "ibm_kingston"
SESSION1_JOB = "dbi9i95ep8hs73andgs0"   # main job of notebook 03 (session 1)
RESUME_JOB = None                      # a session-2 job id, to fetch a job submitted earlier (real runs only)
MODELS = {"LA": "toniot8_r100_Q_s3_LA_seed0",          # replication, layouts A and B
          "CE235": "toniot8_r100_Q_s2.35_CE_seed0",    # H5 test (post-hoc), layout A
          "QNE": "toniot8_r100_Q-noent_s3_LA_seed0"}   # entanglement-free, layout A
POSTHOC = ("CE235",)
SAFETY = 0.8                           # use at most 80% of the remaining QPU seconds
if DRY_RUN:
    RESUME_JOB = None

In [ ]:
import json, time, numpy as np, pandas as pd
from bbd import config, hardware as hw, runner, store
RESULTS = config.RESULTS_DIR
runner.carry_forward(RESULTS, config.INPUT_ROOT, kinds=("ledger",))
missing = [r for r in MODELS.values() if not store.has(RESULTS, "ckpt", r)]
assert not missing, f"checkpoints missing: {missing}. Attach the outputs of notebooks 02 and 05."
S1 = None
try:
    S1 = hw.find_job(SESSION1_JOB)          # path and settings of session 1
    print("session 1:", S1["submitted"], "path", S1["path"], "shots", S1["shots"])
except FileNotFoundError as e:
    print("session 1 record not found (attach the output of notebook 03):", e)
SAVED = hw.find_job(RESUME_JOB) if RESUME_JOB else None
if DRY_RUN:
    service, backend = None, hw.get_backend(name=BACKEND, fake=True)
    remaining = None
else:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    try:
        instance = sec.get_secret("IBM_QUANTUM_INSTANCE")
    except Exception:
        instance = None
    service = hw.connect(sec.get_secret("IBM_QUANTUM_TOKEN"), instance)
    backend = service.backend(BACKEND)
    remaining = hw.remaining_seconds(service)
print(backend.name, "| remaining QPU seconds:", remaining)

In [ ]:
full = {k: hw.model_inputs(RESULTS, rid, 600, 50) for k, rid in MODELS.items()}
keep = hw.every_other(full["LA"]["y_test"])
if DRY_RUN:
    keep = keep[::4]                        # the noisy simulator is slow; a quarter checks the whole pipeline
inputs = {k: hw.subset_inputs(v, keep) for k, v in full.items()}
for k, inp in inputs.items():
    print(k, inp["run_id"], "| s =", inp["s"], "| test", len(inp["y_test"]), "| cal", len(inp["y_cal"]),
          "| max |sim - Qiskit| = %.1e" % hw.verify_against_qiskit(inp))
if SAVED:
    path = list(SAVED["path"])
elif S1 is not None and not DRY_RUN:
    path = list(S1["path"])                 # same physical qubits as session 1
else:
    path = hw.best_path(backend)["path"]
bp = hw.path_info(backend, path)
print("path:", path); print("readout errors today:", np.round(bp["readout"], 4)); print("CZ errors today:", np.round(bp["cz"], 4))
pubs_spec = [("LA", "A"), ("LA", "B"), ("CE235", "A"), ("QNE", "A")]
isa = {}
for k, lay in pubs_spec:
    layout = path if lay == "A" else path[::-1]
    circ = hw.build_circuit(inputs[k]["phi"], inputs[k]["k"], inputs[k]["entangle"], inputs[k]["reupload"])
    isa[(k, lay)] = hw.to_isa(circ, backend, layout)
    chk = hw.check_isa(isa[(k, lay)], circ, layout)
    print(k, lay, {x: chk[x] for x in ("ok", "two_qubit_isa", "depth")}); assert chk["ok"], chk
c0, c1 = hw.calibration_circuits(len(path))
cal_isa = [hw.to_isa(c0, backend, path), hw.to_isa(c1, backend, path)]
config.HW_DIR.mkdir(parents=True, exist_ok=True)
(config.HW_DIR / "path_session2.json").write_text(json.dumps(dict(bp, backend=backend.name, time=time.ctime()), default=str, indent=1))

In [ ]:
n_per_pub = [len(inputs[k]["theta_test"]) + len(inputs[k]["theta_cal"]) for k, _ in pubs_spec]
if SAVED:
    SHOTS = SAVED["shots"]
elif DRY_RUN:
    SHOTS = 128
else:
    SHOTS, est = hw.plan_session_shots(n_per_pub, remaining, cal_shots=8192, safety=SAFETY)
    assert SHOTS is not None, f"not enough QPU time left: need about {est:.0f} s, have {remaining} s"
    print(f"{SHOTS} shots per input, estimated {est:.0f} QPU seconds of {remaining:.0f} remaining")

In [ ]:
if RESUME_JOB is None:
    pubs = [(isa[(k, lay)], np.vstack([inputs[k]["theta_test"], inputs[k]["theta_cal"]]), SHOTS) for k, lay in pubs_spec]
    cal_shots = 2048 if DRY_RUN else 8192
    pubs += [(cal_isa[0], None, cal_shots), (cal_isa[1], None, cal_shots)]
    job = hw.submit(backend, pubs, "session2", dict(order=pubs_spec, shots=SHOTS, path=path, backend=backend.name,
                                                    n_test=600, n_cal=50, keep=keep.tolist(), models=MODELS,
                                                    dry_run=DRY_RUN))
    RESUME_JOB = job.job_id(); print("SESSION-2 JOB ID:", RESUME_JOB, " (saved in hardware/jobs.json)")
else:
    job = service.job(RESUME_JOB)
while not DRY_RUN and str(job.status()) not in ("DONE", "JobStatus.DONE", "ERROR", "CANCELLED"):
    print(time.strftime("%H:%M"), job.status()); time.sleep(120)
res = job.result(); print("usage", hw.job_usage_seconds(job), "s")

In [ ]:
hw_z = {}
for i, (k, lay) in enumerate(pubs_spec):
    nt = len(inputs[k]["theta_test"])
    z = hw.counts_to_z(res[i].data.c, inputs[k]["k"])
    hw_z[(k, lay)] = (z[:nt], z[nt:])
n = len(pubs_spec)
cal = {"z0": hw.counts_to_z(res[n].data.c, len(path))[0], "z1": hw.counts_to_z(res[n + 1].data.c, len(path))[0]}
OUT2 = config.HW_DIR / "session2"
out = hw.analyse(inputs, hw_z, path, cal, SHOTS, posthoc_keys=POSTHOC, out=OUT2)
display(pd.DataFrame({nm: {c: round(v["macro_f1"], 3) for c, v in cs.items()} for nm, cs in out["conditions"].items()}).T)

## Tests

In [ ]:
rep = {"shots": SHOTS, "job": RESUME_JOB}
y = inputs["LA"]["y_test"]
# 1. shot-noise band: is the hardware loss larger than finite sampling alone explains?
rep["shot_band"] = {f"{k}/{lay}": hw.shot_band(inputs[k], SHOTS, k in POSTHOC) for k, lay in pubs_spec}
# 2. paired layout test: same session, same inputs
pa = hw._decisions(inputs["LA"], hw_z[("LA", "A")][0], False)
pb = hw._decisions(inputs["LA"], hw_z[("LA", "B")][0], False)
rep["layout_A_vs_B"] = hw.paired_mcnemar(pa, pb, y)
# 3. Lemma 3 / H5: classes predicted unreachable after the measured contraction, against what happened
alpha = out["fits"]["CE235 + post-hoc / layout A"]["alpha_global"]
rep["h5_prediction"] = hw.h5_prediction(inputs["CE235"], alpha)
c = out["conditions"]["CE235 + post-hoc / layout A"]
rep["h5_observed"] = {cond: dict(never_predicted=c[cond]["never_predicted"], tail_fpr=c[cond]["tail_fpr"],
                                 recall_most_frequent=c[cond]["recall"][0], macro_f1=c[cond]["macro_f1"])
                      for cond in ("sim_exact", "raw", "global", "readout")}
# 4. entanglement-free circuit: hardware loss with and without CZ gates, same layout
cq, cn = out["conditions"]["LA / layout A"], out["conditions"]["QNE / layout A"]
rep["cz_vs_readout"] = dict(Q_loss=cq["sim_exact"]["macro_f1"] - cq["raw"]["macro_f1"],
                            Qnoent_loss=cn["sim_exact"]["macro_f1"] - cn["raw"]["macro_f1"])
print(json.dumps(rep, indent=1, default=float))

In [ ]:
# 5. session 1 against session 2 on the same 304 inputs (fetching a finished job costs no QPU time)
if not DRY_RUN and S1 is not None:
    r1 = service.job(SESSION1_JOB).result()
    order1 = [tuple(o) for o in S1["order"]]
    for lay in ("A", "B"):
        i = order1.index(("LA", lay))
        z1 = hw.counts_to_z(r1[i].data.c, full["LA"]["k"])[:600][keep]
        p1 = hw._decisions(inputs["LA"], z1, False)
        p2 = hw._decisions(inputs["LA"], hw_z[("LA", lay)][0], False)
        rep[f"session1_vs_2_layout_{lay}"] = hw.paired_mcnemar(p1, p2, y)
    print(json.dumps({k: v for k, v in rep.items() if k.startswith("session")}, indent=1))
OUT2.mkdir(parents=True, exist_ok=True)
(OUT2 / "hw2_tests.json").write_text(json.dumps(rep, indent=1, default=float))
print("saved", OUT2 / "hw2_tests.json")